<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h1 align="center">Adaline: Multi-Class Classification with the Delta Rule</h1>

**Adaline** (*ADAptive LInear NEuron*) was introduced in **1960** by *Bernard Widrow* and *Marcian Hoff*.  
It looks like a perceptron, but it learns in a different way: the error is measured on the **linear net input**, **before** any threshold is applied. This makes training a smooth minimisation of the **Mean Squared Error (MSE)**, known as the **LMS** or **Widrow–Hoff** rule.

<br>

<div align="center">
    <img src="images/adaline.png" width="750">
    <br>
    <em><b>Adaline: the error is taken from the linear output, the threshold is used only to classify</b></em>
</div>

<br>

- The neuron computes **net = w₁x₁ + w₂x₂ + b**.
- During training the **target t ∈ {−1, +1}** is compared with **net itself**: **error = t − net**.
- The weights follow the **delta rule**, which moves them down the gradient of the squared error.
- After training, the **sign of net** gives the class.
- A single Adaline separates **two** classes, so for **three** classes we train **three** Adalines, one per class (**one-vs-rest**).

</div>

<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">The Data</h2>

Three groups of random 2-D points (x₁, x₂) are generated from **Gaussian** distributions, all with standard deviation **0.2**:

| Class | Points | Mean of x₁ | Mean of x₂ | Std of x₁ and x₂ |
|---|---|---|---|---|
| Class 1 | 100 | 1.0 | 1.0 | 0.2 |
| Class 2 | 100 | 2.5 | 2.0 | 0.2 |
| Class 3 | 200 | 1.0 | 3.0 | 0.2 |

The three groups are far apart, so each class can be separated from the other two by a straight line.

<div align="center">
    <img src="images/data.png" width="550">
</div>

</div>

In [1]:
# Generate the three Gaussian classes
import random

random.seed(42)   # fixed seed, so every run gives the same data

def make_class(n, mean1, mean2, std=0.2):
    return [(random.gauss(mean1, std), random.gauss(mean2, std)) for _ in range(n)]

CLASS_SPECS = [("Class 1", 100, 1.0, 1.0),
               ("Class 2", 100, 2.5, 2.0),
               ("Class 3", 200, 1.0, 3.0)]

X, labels = [], []          # X: list of (x1, x2), labels: 0, 1 or 2
for k, (name, n, m1, m2) in enumerate(CLASS_SPECS):
    points = make_class(n, m1, m2)
    X += points
    labels += [k] * n

# Check the generated data
def mean(values):
    return sum(values) / len(values)

def std(values):
    m = mean(values)
    return (sum((v - m) ** 2 for v in values) / len(values)) ** 0.5

if __name__ == "__main__":
    print("Class    |   N | mean x1 | mean x2 | std x1 | std x2")
    print("---------------------------------------------------")

    for k, (name, n, m1, m2) in enumerate(CLASS_SPECS):
        xs = [p[0] for p, l in zip(X, labels) if l == k]
        ys = [p[1] for p, l in zip(X, labels) if l == k]
        print(f"{name} | {len(xs):3} | {mean(xs):7.3f} | {mean(ys):7.3f} | {std(xs):6.3f} | {std(ys):6.3f}")

Class    |   N | mean x1 | mean x2 | std x1 | std x2
---------------------------------------------------
Class 1 | 100 |   1.029 |   1.018 |  0.186 |  0.183
Class 2 | 100 |   2.519 |   2.000 |  0.220 |  0.209
Class 3 | 200 |   0.986 |   2.986 |  0.196 |  0.202


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Network Design</h2>

The goal is a **single-layer network** that separates the three classes.

- **Inputs:** the two coordinates **x₁** and **x₂**, plus a constant **1** for the bias.
- **Outputs:** **three Adaline neurons**, one per class. Neuron k answers the question *"does this point belong to class k?"*
- **Connections:** every input is connected to every output neuron, so the network has **2 × 3 = 6 weights** and **3 biases**.
- **Decision rule:** the predicted class is the neuron with the **largest net input**.

**Why is one layer enough?** Each class can be cut off from the other two by a straight line (the classes are *linearly separable*, see the data figure in the previous section), and a single Adaline draws exactly one straight line.

<div align="center">
    <img src="images/network.png" width="650">
    <br>
    <em><b>Single-layer network: 2 inputs, 3 Adaline output neurons</b></em>
</div>

</div>

In [2]:
# Architecture of the network: 2 inputs -> 3 Adaline output neurons (single layer)
N_INPUTS = 2
N_CLASSES = 3

if __name__ == "__main__":
    n_weights = N_INPUTS * N_CLASSES
    n_biases = N_CLASSES

    print("Layer   | Size")
    print("---------------")
    print(f"Input   | {N_INPUTS}  (x1, x2)")
    print(f"Output  | {N_CLASSES}  (one Adaline per class)")
    print(f"\nWeights = {n_weights}, biases = {n_biases}, total parameters = {n_weights + n_biases}")

Layer   | Size
---------------
Input   | 2  (x1, x2)
Output  | 3  (one Adaline per class)

Weights = 6, biases = 3, total parameters = 9


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Targets: One-vs-Rest</h2>

Neuron **k** must answer one question: *"does this point belong to class k?"*

- **t = +1** for the points of class k
- **t = −1** for all other points

So the three neurons see the **same inputs** but have **different targets**.

</div>

In [3]:
# Bipolar targets for neuron k (one-vs-rest)
def one_vs_rest(labels, k):
    return [1 if label == k else -1 for label in labels]

TARGETS = [one_vs_rest(labels, k) for k in range(N_CLASSES)]

if __name__ == "__main__":
    print("Neuron | targets = +1 | targets = -1")
    print("------------------------------------")

    for k, t in enumerate(TARGETS):
        print(f"   {k+1}   | {t.count(1):12} | {t.count(-1):12}")

Neuron | targets = +1 | targets = -1
------------------------------------
   1   |          100 |          300
   2   |          100 |          300
   3   |          200 |          200


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">The Delta Rule (LMS)</h2>

For every training point **(x₁, x₂)** with target **t**:

1. Compute **net = w₁x₁ + w₂x₂ + b**.
2. Compute the error **e = t − net**.
3. Update the weights and the bias:  
   - **wᵢ ← wᵢ + α · e · xᵢ**  
   - **b ← b + α · e**

Unlike the perceptron, the weights change **after every point**, even when its sign is already correct, because the goal is to make **net** itself close to **t**.  
After each epoch we measure the **MSE = mean( (t − net)² )** to follow the learning.

</div>

In [4]:
# Adaline training with the delta rule (pure Python)
def net_input(x1, x2, w1, w2, b):
    return w1*x1 + w2*x2 + b

def mse(X, targets, w1, w2, b):
    return sum((t - net_input(x1, x2, w1, w2, b)) ** 2
               for (x1, x2), t in zip(X, targets)) / len(X)

def train_adaline(X, targets, alpha=0.00009, epochs=1000, init=(0.0, 0.0, 0.0)):

    w1, w2, b = init
    history = []                          # MSE after every epoch

    for epoch in range(epochs):
        for (x1, x2), t in zip(X, targets):
            error = t - net_input(x1, x2, w1, w2, b)
            w1 += alpha * error * x1
            w2 += alpha * error * x2
            b  += alpha * error

        history.append(mse(X, targets, w1, w2, b))

        if history[-1] > 1e6:             # the training is blowing up
            break

    return (w1, w2, b), history


# Final class = the neuron with the largest net input
def classify(x1, x2, weights):
    nets = [net_input(x1, x2, *w) for w in weights]
    return nets.index(max(nets))

<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Training the Three Adalines</h2>

The experiment uses these parameters:

- Learning rate **α = 0.00009**
- **1000** epochs
- Random initial weights and bias in **[0, 1)**

<div align="center">
    <img src="images/mse_curve.png" width="650">
    <br>
    <em><b>MSE of each neuron during training</b></em>
</div>

</div>

In [5]:
# Train one Adaline for every class
if __name__ == "__main__":
    weights, histories = [], []

    for k in range(N_CLASSES):
        init = (random.random(), random.random(), random.random())
        w, history = train_adaline(X, TARGETS[k], alpha=0.00009, epochs=1000, init=init)
        weights.append(w)
        histories.append(history)

    print("MSE during training:\n")
    print("Epoch  | Neuron 1 | Neuron 2 | Neuron 3")
    print("---------------------------------------")
    for e in [1, 10, 100, 500, 1000]:
        print(f"{e:6} | {histories[0][e-1]:8.4f} | {histories[1][e-1]:8.4f} | {histories[2][e-1]:8.4f}")

    print("\nFinal weights:\n")
    print("Neuron |    w1     w2      b")
    print("------------------------------")
    for k, (w1, w2, b) in enumerate(weights):
        print(f"   {k+1}   | {w1:7.3f} {w2:7.3f} {b:7.3f}")

MSE during training:

Epoch  | Neuron 1 | Neuron 2 | Neuron 3
---------------------------------------
     1 |   3.3729 |   5.6333 |   3.6398
    10 |   0.3728 |   0.3690 |   1.0726
   100 |   0.2603 |   0.2415 |   0.1676
   500 |   0.0901 |   0.0956 |   0.0727
  1000 |   0.0576 |   0.0675 |   0.0576

Final weights:

Neuron |    w1     w2      b
------------------------------
   1   |  -0.531  -0.899   2.235
   2   |   1.147  -0.050  -1.958
   3   |  -0.656   0.910  -1.124


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Decision Lines</h2>

Each neuron separates its class with the line **net = 0**:

**w₁x₁ + w₂x₂ + b = 0**  (or **x₂ = −(w₁/w₂)·x₁ − b/w₂** when w₂ is not close to 0)

The three lines divide the plane, and the region where a neuron has the **largest net** is the region of its class.

<div align="center">
    <img src="images/decision_lines.png" width="650">
</div>

</div>

In [6]:
# Line equation and accuracy of every neuron (sign of net = predicted side)
if __name__ == "__main__":
    print("Neuron | Decision line (net = 0)          | Accuracy (vs rest)")
    print("-------------------------------------------------------------")

    for k, (w1, w2, b) in enumerate(weights):
        correct = sum((1 if net_input(x1, x2, w1, w2, b) >= 0 else -1) == t
                      for (x1, x2), t in zip(X, TARGETS[k]))
        print(f"   {k+1}   | {w1:+.3f} x1 {w2:+.3f} x2 {b:+.3f} = 0 | {correct / len(X):8.1%}")

Neuron | Decision line (net = 0)          | Accuracy (vs rest)
-------------------------------------------------------------
   1   | -0.531 x1 -0.899 x2 +2.235 = 0 |   100.0%
   2   | +1.147 x1 -0.050 x2 -1.958 = 0 |   100.0%
   3   | -0.656 x1 +0.910 x2 -1.124 = 0 |   100.0%


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Final Classification</h2>

To label a point we compare the **three net inputs** and choose the largest one.  
The **confusion matrix** counts, for every true class (rows), which class the network predicted (columns).

</div>

In [7]:
# Confusion matrix and overall accuracy
if __name__ == "__main__":
    predictions = [classify(x1, x2, weights) for x1, x2 in X]

    matrix = [[0] * N_CLASSES for _ in range(N_CLASSES)]
    for true, pred in zip(labels, predictions):
        matrix[true][pred] += 1

    print("Confusion Matrix (rows = true class, columns = predicted class):\n")
    print("          Pred 1  Pred 2  Pred 3")
    print("--------------------------------")
    for k, row in enumerate(matrix):
        print(f"True {k+1}  {row[0]:6}  {row[1]:6}  {row[2]:6}")

    correct = sum(matrix[k][k] for k in range(N_CLASSES))
    print(f"\nAccuracy = {correct} / {len(X)} = {correct / len(X):.1%}")

Confusion Matrix (rows = true class, columns = predicted class):

          Pred 1  Pred 2  Pred 3
--------------------------------
True 1     100       0       0
True 2       0     100       0
True 3       0       0     200

Accuracy = 400 / 400 = 100.0%


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2 align="left">Effect of the Learning Rate</h2>

The learning rate **α** controls the size of every step, so it changes both the **training time** and the **final result**:

- **Too small:** learning is very slow, and the MSE is still high after the last epoch.
- **Good:** the MSE falls quickly and settles at its minimum.
- **Large:** the weights keep jumping around the minimum and follow the **last points they saw**. The points are shown class by class, so Class 3 (the last one) dominates the final weights.
- **Too large:** every step overshoots the minimum, the error **grows** instead of shrinking, and training **diverges**.

The table trains all three neurons for **1000 epochs** with each α (the same initial values every time).  
*Epochs to MSE ≤ 0.1* is the first epoch in which the **mean MSE** of the three neurons drops to 0.1 or below.

</div>

In [8]:
# Try different learning rates
LEARNING_RATES = [0.000001, 0.00009, 0.001, 0.01, 0.1, 0.3]
lr_results = {}      # alpha -> weights, mean MSE per epoch, accuracy

if __name__ == "__main__":
    print("   alpha   | epochs to MSE <= 0.1 | final MSE | accuracy | diverged")
    print("-------------------------------------------------------------------")

    for alpha in LEARNING_RATES:
        trained, curves = [], []

        for k in range(N_CLASSES):
            rng = random.Random(k)                      # same start for every alpha
            init = (rng.random(), rng.random(), rng.random())
            w, history = train_adaline(X, TARGETS[k], alpha=alpha, epochs=1000, init=init)
            trained.append(w)
            curves.append(history)

        n_epochs = min(len(h) for h in curves)          # shorter if a neuron blew up
        mean_curve = [mean([h[e] for h in curves]) for e in range(n_epochs)]
        acc = sum(classify(x1, x2, trained) == l for (x1, x2), l in zip(X, labels)) / len(X)
        diverged = mean_curve[-1] > 1e6
        lr_results[alpha] = {"weights": trained, "curve": mean_curve, "accuracy": acc}

        reached = [e + 1 for e, v in enumerate(mean_curve) if v <= 0.1]
        first = reached[0] if reached else "never"
        final = "huge" if diverged else f"{mean_curve[-1]:.4f}"
        print(f"{alpha:10g} | {first:>20} | {final:>9} | {acc:8.1%} | {'yes' if diverged else 'no'}")

   alpha   | epochs to MSE <= 0.1 | final MSE | accuracy | diverged
-------------------------------------------------------------------
     1e-06 |                never |    0.7980 |    50.5% | no
     9e-05 |                  472 |    0.0619 |   100.0% | no
     0.001 |                   47 |    0.0586 |   100.0% | no
      0.01 |                never |    0.1197 |    99.8% | no
       0.1 |                never |    0.6235 |    56.2% | no
       0.3 |                never |      huge |    25.0% | yes


<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

The figures show the same experiment: the **learning curves** for every α, and the **decision lines** for a very small, a good and a large α (each dashed line has the colour of its class).

<div align="center">
    <img src="images/lr_curves.png" width="650">
    <br>
    <em><b>Mean MSE of the three neurons for different learning rates</b></em>
</div>

<br>

<div align="center">
    <img src="images/lr_lines.png" width="1000">
    <br>
    <em><b>Decision lines after 1000 epochs</b></em>
</div>

</div>

<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2>Conclusion</h2>
In this notebook, we trained three **Adaline** neurons, with only pure Python, to classify three groups of 2-D points.

Through our experiments, we demonstrated:
- **Learning from the Linear Output:** The delta rule minimises the MSE by correcting the weights after every point. With α = 0.00009 and 1000 epochs, all three neurons reached a low and stable MSE.
- **One-vs-Rest Classification:** Three neurons, one per class, were enough to separate the three groups. Choosing the neuron with the largest net input classified **all 400 points** correctly.
- **The Role of α:** A very small α learns too slowly (50.5% accuracy for α = 10⁻⁶), values between about 10⁻⁴ and 10⁻³ work best, α = 0.1 is already too noisy (56.2%), and α = 0.3 makes the training **diverge**.

Adaline is the starting point of **gradient descent**, the same idea that trains modern deep networks.

</div>

<div style="border:1px solid #d0d7de; padding:16px; border-radius:8px;">

<h2>👩‍💻 About the Author</h2>

<b>Sarah M Afshar</b>  
<i>Python Developer | Generative AI Specialist</i>

I am a passionate developer focused on the intersection of <b>Large Language Models (LLMs)</b>, neural network architectures, and the evolution of intelligent systems.  
This project is part of my journey in exploring the mathematical foundations of AI.

🔗 GitHub: https://github.com/afshars  

🔗 LinkedIn: https://www.linkedin.com/in/sara-m-afshar/

</div>